In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# Path settings (matching your original code)
desktop = os.getcwd()
feature_path = os.path.join(desktop, "gs-homo-lumo-features.xlsx")
raw_path = os.path.join(desktop, "gs-homo-lumo.xlsx")
save_folder = os.path.join(desktop, "PAPER_SVR_HOLDOUT")
os.makedirs(save_folder, exist_ok=True)

# Read data
df_features = pd.read_excel(feature_path)
df_raw = pd.read_excel(raw_path)

TARGET = "E"
df_raw = df_raw.dropna(subset=[TARGET])

exclude_cols = ["Index", "SMILES", "HOMO", "LUMO", "ΔEH-L"]
ALL_FEATURES = [c for c in df_features.columns if c not in exclude_cols]

# Merge features and target (retain the index column)
df = pd.merge(df_features, df_raw[["Index", TARGET]], on="Index")
df = df.dropna(subset=[TARGET])

# Split parameters (exactly as in your code)
HOLD_OUT_SIZE = 0.2
RANDOM_STATE_SPLIT = 13

# Prepare data for splitting: features, target, index
X_raw = df[ALL_FEATURES].values
y_raw = df[TARGET].values
index_raw = df["Index"].values

# Split features, target and index together
X_dev, X_hold, y_dev, y_hold, idx_dev, idx_hold = train_test_split(
    X_raw, y_raw, index_raw,
    test_size=HOLD_OUT_SIZE,
    random_state=RANDOM_STATE_SPLIT,
    shuffle=True
)

# Build the development DataFrame (including index, all features and target)
dev_df = pd.DataFrame(X_dev, columns=ALL_FEATURES)
dev_df["Index"] = idx_dev
dev_df[TARGET] = y_dev
# Move the index column to the first position (optional)
cols = ["Index"] + ALL_FEATURES + [TARGET]
dev_df = dev_df[cols]

# Build the holdout test set DataFrame
hold_df = pd.DataFrame(X_hold, columns=ALL_FEATURES)
hold_df["Index"] = idx_hold
hold_df[TARGET] = y_hold
hold_df = hold_df[cols]

# Save
dev_path = os.path.join(save_folder, "Development_Set.xlsx")
hold_path = os.path.join(save_folder, "Holdout_Test_Set.xlsx")
dev_df.to_excel(dev_path, index=False)
hold_df.to_excel(hold_path, index=False)

print(f"Development set (training set) saved to：{dev_path}  (Sample count：{len(dev_df)})")
print(f"Holdout test set saved to：{hold_path}  (Sample count：{len(hold_df)})")

Development set (training set) saved to：D:\MLcof\github\final\PAPER_SVR_HOLDOUT\Development_Set.xlsx  (Sample count：56)
Holdout test set saved to：D:\MLcof\github\final\PAPER_SVR_HOLDOUT\Holdout_Test_Set.xlsx  (Sample count：15)


In [2]:
# ===================== PAPER_SVR_HOLDOUT_FINAL.py =====================
# Revised version: correct use of a holdout test set + repeated hold-out validation + SHAP analysis
# For a single data source, with accurate terminology and no data leakage

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib
import os
import time
import warnings
import random

# ===================== Set all random seeds =====================
np.random.seed(42)
random.seed(42)

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from bayes_opt import BayesianOptimization

warnings.filterwarnings("ignore")

# ===================== Global font settings (Arial, size 18) =====================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 18
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['axes.unicode_minus'] = False

# Check whether Arial is available
import matplotlib.font_manager as fm
if not any('Arial' in f.name for f in fm.fontManager.ttflist):
    print("⚠️ Arial font not found; use the default sans-serif font")
    plt.rcParams['font.family'] = 'sans-serif'

# ===================== Path configuration =====================
desktop = os.getcwd()
save_folder = os.path.join(desktop, "PAPER_SVR_HOLDOUT")
os.makedirs(save_folder, exist_ok=True)

# ===================== Data loading =====================
print("\n📥 Load data...")
feature_path = os.path.join(desktop, "gs-homo-lumo-features.xlsx")
raw_path = os.path.join(desktop, "gs-homo-lumo.xlsx")

df_features = pd.read_excel(feature_path)
df_raw = pd.read_excel(raw_path)

TARGET = "E"
df_raw = df_raw.dropna(subset=[TARGET])

exclude_cols = ["Index", "SMILES", "HOMO", "LUMO", "ΔEH-L"]
ALL_FEATURES = [c for c in df_features.columns if c not in exclude_cols]

df = pd.merge(df_features, df_raw[["Index", TARGET]], on="Index")
df = df.dropna(subset=[TARGET])

X_raw = df[ALL_FEATURES].values
y_raw = df[TARGET].values

print(f"Original sample count: {len(X_raw)}, Number of features: {X_raw.shape[1]}")

# ===================== Step 1: split off the hold-out test set =====================
HOLD_OUT_SIZE = 0.2          # Reserve 20% for final evaluation
RANDOM_STATE_SPLIT = 13

X_dev, X_hold, y_dev, y_hold = train_test_split(
    X_raw, y_raw, test_size=HOLD_OUT_SIZE, random_state=RANDOM_STATE_SPLIT, shuffle=True
)

# Clean the development and holdout test sets separately (avoid data leakage)
def clean_data(X, y):
    mask = ~(np.isnan(X).any(axis=1) | np.isinf(X).any(axis=1))
    mask &= ~(np.isnan(y) | np.isinf(y))
    return X[mask], y[mask]

X_dev, y_dev = clean_data(X_dev, y_dev)
X_hold, y_hold = clean_data(X_hold, y_hold)

print(f"\n📌 Development set (Used for training + validation): {len(X_dev)} samples")
print(f"📌 Holdout test set (Final evaluation): {len(X_hold)} samples")

# ===================== Bayesian optimization function =====================
def optimize_svr(X_train, y_train, init_points=5, n_iter=10, seed=42):
    def cv_score(C, epsilon, gamma):
        try:
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("svr", SVR(kernel='rbf', C=C, epsilon=epsilon, gamma=gamma))
            ])
            cv = KFold(n_splits=5, shuffle=True, random_state=42)
            scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2', error_score=np.nan)
            scores = scores[~np.isnan(scores)]
            return np.mean(scores) if len(scores) > 0 else -1e6
        except:
            return -1e6

    opt = BayesianOptimization(
        f=cv_score,
        pbounds={
            "C": (1, 20),
            "epsilon": (0.005, 0.7),
            "gamma": (0.0025, 0.7)
        },
        random_state=seed,
        verbose=0
    )
    opt.maximize(init_points=init_points, n_iter=n_iter)
    return opt.max["params"]

# ===================== Repeated hold-out validation within the development set =====================
N_REPEATS = 15          # Repeat 15 times to assess stability
INNER_TEST_SIZE = 0.1   # Split off 10% of the development set for internal testing each time

all_train_r2 = []
all_internal_test_r2 = []
all_train_rmse = []
all_internal_test_rmse = []
all_nested_cv_r2 = []

print(f"\n🚀 Perform on the development set: {N_REPEATS} repetitions of hold-out validation...")

for rep in range(1, N_REPEATS + 1):
    seed_rep = rep * 100
    X_train, X_test, y_train, y_test = train_test_split(
        X_dev, y_dev, test_size=INNER_TEST_SIZE, random_state=seed_rep, shuffle=True
    )

    # Nested CV Evaluate the stability of hyperparameter optimization
    outer_cv = KFold(n_splits=5, shuffle=True, random_state=seed_rep)
    nested_r2 = []
    for fold_idx, (tr_idx, val_idx) in enumerate(outer_cv.split(X_train)):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]
        best_params = optimize_svr(X_tr, y_tr, init_points=5, n_iter=10, seed=seed_rep + fold_idx)
        model = Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel='rbf', **best_params))])
        model.fit(X_tr, y_tr)
        nested_r2.append(r2_score(y_val, model.predict(X_val)))
    mean_nested_r2 = np.mean(nested_r2)
    all_nested_cv_r2.append(mean_nested_r2)

    # Optimize the final model on the current training set
    final_params = optimize_svr(X_train, y_train, init_points=8, n_iter=15, seed=seed_rep + 200)
    model = Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel='rbf', **final_params))])
    model.fit(X_train, y_train)

    # Training set performance
    y_train_pred = model.predict(X_train)
    train_r2 = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

    # Internal test set performance
    y_test_pred = model.predict(X_test)
    test_r2 = r2_score(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    all_train_r2.append(train_r2)
    all_internal_test_r2.append(test_r2)
    all_train_rmse.append(train_rmse)
    all_internal_test_rmse.append(test_rmse)

    print(f"[{rep:02d}] Nested CV R²={mean_nested_r2:.3f} | Train R²={train_r2:.3f} | Test R²={test_r2:.3f}")

print("\n" + "="*60)
print("Repeated validation statistics within the development set (15 random splits):")
print(f"  Nested CV R²   = {np.mean(all_nested_cv_r2):.4f} ± {np.std(all_nested_cv_r2):.4f}")
print(f"  Train R²       = {np.mean(all_train_r2):.4f} ± {np.std(all_train_r2):.4f}")
print(f"  Internal Test R² = {np.mean(all_internal_test_r2):.4f} ± {np.std(all_internal_test_r2):.4f}")
print("="*60)

# ===================== Final model: train on the entire development set =====================
print("\n📌 Train the final model on the entire development set...")
final_params = optimize_svr(X_dev, y_dev, init_points=10, n_iter=20, seed=2025)
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel='rbf', **final_params))
])
final_model.fit(X_dev, y_dev)

# Performance on the development set
y_dev_pred = final_model.predict(X_dev)
dev_r2 = r2_score(y_dev, y_dev_pred)
dev_rmse = np.sqrt(mean_squared_error(y_dev, y_dev_pred))
dev_mae = mean_absolute_error(y_dev, y_dev_pred)

# Final evaluation on the holdout test set (generalization to truly unseen data)
y_hold_pred = final_model.predict(X_hold)
hold_r2 = r2_score(y_hold, y_hold_pred)
hold_rmse = np.sqrt(mean_squared_error(y_hold, y_hold_pred))
hold_mae = mean_absolute_error(y_hold, y_hold_pred)

print("\n📊 Final model performance:")
print(f"  Development set (Training)   : R²={dev_r2:.4f}, RMSE={dev_rmse:.4f}, MAE={dev_mae:.4f}")
print(f"  Holdout test set (Final evaluation): R²={hold_r2:.4f}, RMSE={hold_rmse:.4f}, MAE={hold_mae:.4f}")

# ===================== Bootstrap confidence interval (holdout test set R²) =====================
n_boot = 1000
boot_r2 = []
for _ in range(n_boot):
    idx = np.random.choice(len(y_hold), len(y_hold), replace=True)
    boot_r2.append(r2_score(y_hold[idx], y_hold_pred[idx]))
ci_low, ci_high = np.percentile(boot_r2, [2.5, 97.5])
print(f"\n95% Bootstrap CI for holdout test set R²: [{ci_low:.4f}, {ci_high:.4f}]")

# ===================== Y-scrambling test (on the development set) =====================
print("\n🧪 Y-scrambling test...")
scramble_r2 = []
for _ in range(50):
    y_shuffled = np.random.permutation(y_dev)
    model = Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel='rbf', **final_params))])
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_dev, y_shuffled, cv=cv, scoring='r2')
    scramble_r2.append(np.mean(scores))
print(f"Mean after shuffling the target values: R² = {np.mean(scramble_r2):.4f} ± {np.std(scramble_r2):.4f} (Should be near 0 or negative)")

# ===================== SHAP analysis (using the development set) =====================
print("\n🔎 SHAP feature importance analysis (based on the development set)...")
background = shap.sample(X_dev, min(100, len(X_dev)))
explainer = shap.KernelExplainer(final_model.predict, background)
shap_values = explainer.shap_values(X_dev)

importance = np.abs(shap_values).mean(axis=0)
imp_df = pd.DataFrame({"feature": ALL_FEATURES, "importance": importance})
imp_df = imp_df.sort_values("importance", ascending=False)
imp_df.to_excel(os.path.join(save_folder, "SHAP_Feature_Importance.xlsx"), index=False)

# Top 15 features
top_n = 13
top_features = imp_df.head(top_n)["feature"].tolist()
top_indices = [ALL_FEATURES.index(f) for f in top_features]

# Bar plot (7.2×7.2 cm)
fig, ax = plt.subplots(figsize=(7.2/2.54, 6.1/2.54))
shap.summary_plot(shap_values[:, top_indices], X_dev[:, top_indices],
                  feature_names=top_features, show=False, plot_type="bar", max_display=top_n)
ax.tick_params(axis='both', labelsize=16)
for label in ax.get_yticklabels():
    label.set_fontsize(18)
plt.tight_layout()
plt.savefig(os.path.join(save_folder, "SHAP_Top15_Bar.png"), dpi=300, bbox_inches="tight")
plt.close()

# Dot plot
fig, ax = plt.subplots(figsize=(6.39/2.54, 6.55/2.54))
shap.summary_plot(shap_values[:, top_indices], X_dev[:, top_indices],
                  feature_names=top_features, show=False, max_display=top_n, cmap="coolwarm")
ax.tick_params(axis='both', labelsize=18)
for label in ax.get_yticklabels():
    label.set_fontsize(18)
    ax.set_xlabel("SHAP value", fontsize=18)
plt.tight_layout()
plt.savefig(os.path.join(save_folder, "SHAP_Top15_Dot.png"), dpi=300, bbox_inches="tight")
plt.close()
print(f"✅ SHAP plots saved (top {top_n}features)")

# ===================== Plot: development and holdout test set scatter plot (6.1 x 6.1 inches) =====================
plt.figure(figsize=(6.39, 6.47))
# Development set (blue, no black edges)
plt.scatter(y_dev, y_dev_pred, c='#00AEED', alpha=0.9, label=f'Train (R²={dev_r2:.3f})')
# Holdout test set (red, no black edges)
plt.scatter(y_hold, y_hold_pred, c='#B91225', alpha=0.9, label=f'Test (R²={hold_r2:.3f})')

# Diagonal line
min_val = min(y_dev.min(), y_hold.min(), y_dev_pred.min(), y_hold_pred.min())
max_val = max(y_dev.max(), y_hold.max(), y_dev_pred.max(), y_hold_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--', lw=1.5)

# ===================== Key: use exactly the same limits for both axes =====================
plt.xlim(min_val, max_val)   # X-axis limits
plt.ylim(min_val, max_val)   # Y-axis limits are exactly the same as the X-axis limits

plt.xlabel("Calculated ΔEg", fontsize=18)
plt.ylabel("Predicted ΔEg", fontsize=18)
plt.legend(loc='best', fontsize=18)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(save_folder, "Scatter_Dev_Holdout.png"), dpi=600)
plt.close()

# ===================== Residual plot =====================
plt.figure(figsize=(6.1, 5))
resid_dev = y_dev - y_dev_pred
resid_hold = y_hold - y_hold_pred
plt.scatter(y_dev_pred, resid_dev, c='blue', alpha=0.6, label='Train')
plt.scatter(y_hold_pred, resid_hold, c='red', alpha=0.6, label='Test')
plt.axhline(y=0, color='k', linestyle='--', lw=1.5)
plt.xlabel("Predicted E", fontsize=16)
plt.ylabel("Residuals", fontsize=16)
plt.title("Residual Plot", fontsize=16)
plt.legend(fontsize=16)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(save_folder, "Residual_Plot.png"), dpi=600)
plt.close()

# ===================== Save the model and predictions =====================
joblib.dump(final_model, os.path.join(save_folder, "SVR_final_model.pkl"))
joblib.dump(ALL_FEATURES, os.path.join(save_folder, "features_list.pkl"))

# Save predicted values
dev_pred_df = pd.DataFrame({"Experimental": y_dev, "Predicted": y_dev_pred, "Residual": resid_dev})
hold_pred_df = pd.DataFrame({"Experimental": y_hold, "Predicted": y_hold_pred, "Residual": resid_hold})
dev_pred_df.to_excel(os.path.join(save_folder, "Development_Predictions.xlsx"), index=False)
hold_pred_df.to_excel(os.path.join(save_folder, "Holdout_Predictions.xlsx"), index=False)

# Save repeated validation results
repeat_results = pd.DataFrame({
    "repeat": range(1, N_REPEATS+1),
    "nested_cv_r2": all_nested_cv_r2,
    "train_r2": all_train_r2,
    "train_rmse": all_train_rmse,
    "test_r2": all_internal_test_r2,
    "test_rmse": all_internal_test_rmse
})
repeat_results.to_excel(os.path.join(save_folder, "Repeated_Holdout_Results.xlsx"), index=False)

# ===================== Final summary =====================
print("\n" + "="*60)
print("🎉 All done! Final results summary:")
print(f"  Repeated validation within the development set (15 repetitions) Mean test R² = {np.mean(all_internal_test_r2):.3f} ± {np.std(all_internal_test_r2):.3f}")
print(f"  Final model on the holdout test set: R² = {hold_r2:.3f} (95% CI: [{ci_low:.3f}, {ci_high:.3f}])")
print(f"  Y-scrambling Mean R² = {np.mean(scramble_r2):.3f}")
print(f"📁 All results saved in: {save_folder}")
print("="*60)


📥 Load data...
Original sample count: 71, Number of features: 29

📌 Development set (Used for training + validation): 56 samples
📌 Holdout test set (Final evaluation): 15 samples

🚀 Perform on the development set: 15 repetitions of hold-out validation...
[01] Nested CV R²=0.559 | Train R²=0.969 | Test R²=0.857
[02] Nested CV R²=0.587 | Train R²=0.915 | Test R²=0.676
[03] Nested CV R²=0.627 | Train R²=0.956 | Test R²=0.676
[04] Nested CV R²=0.668 | Train R²=0.971 | Test R²=0.823
[05] Nested CV R²=0.678 | Train R²=0.998 | Test R²=0.796
[06] Nested CV R²=0.590 | Train R²=0.875 | Test R²=0.844
[07] Nested CV R²=0.583 | Train R²=0.934 | Test R²=0.636
[08] Nested CV R²=0.513 | Train R²=0.971 | Test R²=0.867
[09] Nested CV R²=0.327 | Train R²=0.984 | Test R²=0.904
[10] Nested CV R²=0.298 | Train R²=0.889 | Test R²=0.854
[11] Nested CV R²=0.410 | Train R²=0.965 | Test R²=0.837
[12] Nested CV R²=0.231 | Train R²=0.989 | Test R²=0.676
[13] Nested CV R²=0.468 | Train R²=0.964 | Test R²=0.923
[14

  0%|          | 0/56 [00:00<?, ?it/s]

✅ SHAP plots saved (top 13features)

🎉 All done! Final results summary:
  Repeated validation within the development set (15 repetitions) Mean test R² = 0.808 ± 0.092
  Final model on the holdout test set: R² = 0.922 (95% CI: [0.736, 0.953])
  Y-scrambling Mean R² = -0.655
📁 All results saved in: D:\MLcof\github\final\PAPER_SVR_HOLDOUT


In [5]:
# ===================== predict_new_molecules.py =====================
import pandas as pd
import numpy as np
import joblib
import os

# ===================== Paths =====================
desktop = os.getcwd()

model_path = os.path.join(desktop, "PAPER_SVR_HOLDOUT", "SVR_final_model.pkl")
new_data_path = os.path.join(desktop, "new_molecules-features.xlsx")

# ===================== Load the model =====================
loaded = joblib.load(model_path)

# Support both Pipeline and dict formats
if isinstance(loaded, dict):
    model = loaded["model"]
    FEATURE_COLS = loaded["features"]
else:
    model = loaded
    FEATURE_COLS = None

print("✅ Model loaded successfully")
print(f"📌 Model type: {type(model)}")

# ===================== Load new data =====================
df_new = pd.read_excel(new_data_path)

# ===================== Automatic feature processing =====================
if FEATURE_COLS is not None:
    # ✔ Standard mode (recommended)
    
    missing = [col for col in FEATURE_COLS if col not in df_new.columns]
    if len(missing) > 0:
        raise ValueError(f"❌ Missing feature columns: {missing}")

    X_new = df_new[FEATURE_COLS].values

else:
    # ✔ fallback mode (when features were not saved with the Pipeline)
    print("⚠️ FEATURE_COLS not found; attempt automatic alignment...")

    # Try to identify numeric feature columns
    exclude = ["Index", "SMILES", "ID"]
    possible_features = [c for c in df_new.columns if c not in exclude]

    X_new = df_new[possible_features].values

# ===================== Check NaN / inf =====================
mask = ~(np.isnan(X_new).any(axis=1) | np.isinf(X_new).any(axis=1))
X_new = X_new[mask]
df_new = df_new.loc[mask].reset_index(drop=True)

print(f"📊 Valid sample count: {len(X_new)}")

# ===================== Predict =====================
preds = model.predict(X_new)

# ===================== Save results =====================
from decimal import Decimal, ROUND_HALF_UP
df_new["Predicted E"] = [float(Decimal(str(value)).quantize(Decimal("0.0001"), rounding=ROUND_HALF_UP)) for value in preds]

save_path = os.path.join(desktop, "new_molecule_predictions.xlsx")
df_new.to_excel(save_path, index=False)

print("\n🎉 Prediction complete！")
print(f"📁 Save path: {save_path}")

# ===================== Output results =====================
print("\n📊 Prediction results：")
for i, p in enumerate(preds, 1):
    print(f"Molecule {i}: {p:.4f}")

✅ Model loaded successfully
📌 Model type: <class 'sklearn.pipeline.Pipeline'>
⚠️ FEATURE_COLS not found; attempt automatic alignment...
📊 Valid sample count: 1087

🎉 Prediction complete！
📁 Save path: D:\MLcof\github\final\new_molecule_predictions.xlsx

📊 Prediction results：
Molecule 1: 3.8465
Molecule 2: 3.8479
Molecule 3: 3.8583
Molecule 4: 3.7913
Molecule 5: 3.7927
Molecule 6: 3.8502
Molecule 7: 3.8819
Molecule 8: 3.8833
Molecule 9: 3.9352
Molecule 10: 3.9069
Molecule 11: 3.9083
Molecule 12: 3.9213
Molecule 13: 3.6717
Molecule 14: 3.6827
Molecule 15: 3.7537
Molecule 16: 3.6164
Molecule 17: 3.6286
Molecule 18: 3.7203
Molecule 19: 3.3347
Molecule 20: 3.3462
Molecule 21: 3.4904
Molecule 22: 3.5450
Molecule 23: 3.5565
Molecule 24: 3.6870
Molecule 25: 3.7042
Molecule 26: 3.7188
Molecule 27: 3.7319
Molecule 28: 3.4964
Molecule 29: 3.5112
Molecule 30: 3.5269
Molecule 31: 3.2715
Molecule 32: 3.2846
Molecule 33: 3.2988
Molecule 34: 3.1353
Molecule 35: 3.1495
Molecule 36: 3.1649
Molecule 37: 3